# `json2vec` Hello World

This notebook trains a tiny model from an in-memory synthetic dataset.

In [1]:
import lightning.pytorch as lit
import torch
from rich.pretty import pprint

import json2vec as j2v

In [2]:
@j2v.shim(yields=True)
def hello_world_records(observation: dict, strata: j2v.Strata):

    records = [
        {"color": "red", "label": "warm"},
        {"color": "orange", "label": "warm"},
        {"color": "yellow", "label": "warm"},
        {"color": "blue", "label": "cool"},
        {"color": "green", "label": "cool"},
        {"color": "purple", "label": "cool"},
    ]

    yield from records

In [3]:
params = j2v.Hyperparameters(
    d_model=16,
    fields=j2v.Array(
        name="record",
        fields=[
            j2v.Category(name="color", query="[*].color", max_vocab_size=16),
            j2v.Category(name="label", query="[*].label", max_vocab_size=8, topk=[2]),
        ],
    ),
    target=j2v.Address("record", "label"),
    embed=j2v.Address("record"),
)


model = j2v.Architecture(
    hyperparameters=params,
    batch_size=4,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)

datamodule = j2v.StreamingDataModule.from_model(
    model,
    dataset=j2v.Dataset(processor=hello_world_records),
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    file_buffer_size=1,
    observation_buffer_size=32,
    sample_rate=1.0,
)

2026-05-18 20:23:42.884 | INFO     | json2vec.architecture.root:__init__:178 - initialized JSON2Vec module


In [4]:
trainer = lit.Trainer(
    accelerator="mps",
    max_epochs=20,
    logger=False,
    enable_progress_bar=False,
    enable_model_summary=False,
    # enable_checkpointing=False,
)

trainer.fit(model=model, datamodule=datamodule)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /Users/grantham/Desktop/json2vec-oss/examples/checkpoints exists and is not empty.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/grantham/Desktop/j

In [5]:
batch = [[{"color": "red"}], [{"color": "blue"}]]

In [6]:
pprint(model.predict(batch))

{
│   'record/label': {
│   │   'state': {
│   │   │   'valued': [0.997145414352417, 0.9977313876152039],
│   │   │   'null': [0.0009406362078152597, 0.0004456527531147003],
│   │   │   'padded': [0.0009307726286351681, 0.0011621644953265786],
│   │   │   'masked': [0.0005079101538285613, 0.0002518455439712852],
│   │   │   'other': [0.0004754973342642188, 0.00040901225293055177]
│   │   },
│   │   'content': {
│   │   │   'value': ['warm', 'cool'],
│   │   │   'probability': [0.9830919504165649, 0.9903541803359985],
│   │   │   'topk': [
│   │   │   │   [
│   │   │   │   │   {'label': 'warm', 'probability': 0.9830919504165649},
│   │   │   │   │   {'label': 'cool', 'probability': 0.01690816879272461}
│   │   │   │   ],
│   │   │   │   [
│   │   │   │   │   {'label': 'cool', 'probability': 0.9903541803359985},
│   │   │   │   │   {'label': 'warm', 'probability': 0.009645894169807434}
│   │   │   │   ]
│   │   │   ]
│   │   }
│   }
}

In [7]:
pprint(model.embed(batch))

{
│   'record': {
│   │   'embedding': [
│   │   │   [
│   │   │   │   -0.2800813913345337,
│   │   │   │   0.05504139885306358,
│   │   │   │   -0.4645729660987854,
│   │   │   │   0.6124818325042725,
│   │   │   │   -0.27635669708251953,
│   │   │   │   0.03775780275464058,
│   │   │   │   0.20151850581169128,
│   │   │   │   0.2108536809682846,
│   │   │   │   -0.16385257244110107,
│   │   │   │   0.008904410526156425,
│   │   │   │   -0.08384954184293747,
│   │   │   │   -0.17437222599983215,
│   │   │   │   0.20270167291164398,
│   │   │   │   -0.01726674847304821,
│   │   │   │   0.230205699801445,
│   │   │   │   -0.07714295387268066
│   │   │   ],
│   │   │   [
│   │   │   │   -0.09487564116716385,
│   │   │   │   0.26189741492271423,
│   │   │   │   0.4195384085178375,
│   │   │   │   -0.10716091841459274,
│   │   │   │   0.35550060868263245,
│   │   │   │   0.2762298285961151,
│   │   │   │   -0.18970347940921783,
│   │   │   │   -0.35134023427963257,
│   │   │   │   0.21885725855827332,
│   │   │   │   -0.4466915726661682,
│   │   │   │   -0.12528932094573975,
│   │   │   │   0.04716752469539642,
│   │   │   │   0.1954517513513565,
│   │   │   │   -0.20177529752254486,
│   │   │   │   -0.1604788601398468,
│   │   │   │   0.05271846801042557
│   │   │   ]
│   │   ]
│   }
}

In [8]:
model.plot(detail=True)

╭─ JSON2Vec ───────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ n_heads: 4                                                                                                           │
│ d_model: 16                                                                                                          │
│ target: ['record/label']                                                                                             │
│ embed: ['record']                                                                                                    │
│ parameters: 14454                                                                                                    │
│                                                                                                                      │
│   ┏━ record (array) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ │
│   ┃ n_heads: 4                                                                                                     ┃ │
│   ┃ attention: mha                                                                                                 ┃ │
│   ┃ max_length: 1                                                                                                  ┃ │
│   ┃ n_outputs: 1                                                                                                   ┃ │
│   ┃ n_linear: 1                                                                                                    ┃ │
│   ┃ n_layers: 1                                                                                                    ┃ │
│   ┃ parameters: 6608                                                                                               ┃ │
│   ┃                                                                                                                ┃ │
│   ┃   ╭─ color (category) ───────────────────────────────────────────────────────────────────────────────────────╮ ┃ │
│   ┃   │ address: record/color                                                                                    │ ┃ │
│   ┃   │ n_heads: 4                                                                                               │ ┃ │
│   ┃   │ query: [*].color                                                                                         │ ┃ │
│   ┃   │ pooling: query                                                                                           │ ┃ │
│   ┃   │ weight: 1.0                                                                                              │ ┃ │
│   ┃   │ n_linear: 1                                                                                              │ ┃ │
│   ┃   │ max_vocab_size: 16                                                                                       │ ┃ │
│   ┃   │ n_bands: 8                                                                                               │ ┃ │
│   ┃   │ p_unavailable: 0.01                                                                                      │ ┃ │
│   ┃   │ topk: []                                                                                                 │ ┃ │
│   ┃   │ parameters: 4055                                                                                         │ ┃ │
│   ┃   │                                                                                                          │ ┃ │
│   ┃   │ state                                                                                                    │ ┃ │
│   ┃   │   vocabulary_size: 6                                                                                     │ ┃ │
│   ┃   │   vocabulary:                                                                                            │ ┃ │
│   ┃   │     ['orange', 'green', 'red', 'purple', 'yellow',                                                       │ ┃ │
│   ┃  

'<!DOCTYPE html>\n<html>\n<head>\n<meta charset="UTF-8">\n<style>\n\nbody {\n    color: #000000;\n    background-color: #ffffff;\n}\n</style>\n</head>\n<body>\n    <pre style="font-family:Menlo,\'DejaVu Sans Mono\',consolas,\'Courier New\',monospace"><code style="font-family:inherit">╭─ JSON2Vec ───────────────────────────────────────────────────────────────────────────────────────────────────────────╮\n│ n_heads: 4                                                                                                           │\n│ d_model: 16                                                                                                          │\n│ target: [&#x27;record/label&#x27;]                                                                                             │\n│ embed: [&#x27;record&#x27;]                                                                                                    │\n│ parameters: 14454                                                                  